# Digital Filters
This notebook covers the four projects in this folder's README:
1. **Filter design and analysis**: FIR and IIR low-pass, high-pass, band-pass and band-stop filters, with their frequency, phase, group delay and impulse responses, applied to audio.
2. **Real-time filter implementation**: block-by-block processing and filter state.
3. **Filter design comparison**: window, frequency sampling, least squares and Parks-McClellan FIR designs against Butterworth, Chebyshev and elliptic IIR designs, all built to the same specification.
4. **Filter visualization**: poles, zeros and stability.

Start with `filter_design_foundations.ipynb`, which builds an FIR filter by hand from a windowed sinc and derives IIR filters from analog prototypes with the bilinear transform. This notebook uses SciPy's design functions, which automate those recipes.

**FIR vs IIR in one line each:**
- **FIR** (finite impulse response): $y[n] = \sum_k b_k\,x[n-k]$. This is convolution with $h = b$ (see `01_Discrete_Time_Signals_and_Systems`). It is always stable and can have exactly linear phase, but needs many coefficients for a sharp cutoff.
- **IIR** (infinite impulse response): $y[n] = \sum_k b_k\,x[n-k] - \sum_{k\ge1} a_k\,y[n-k]$. Feeding back past outputs gives sharp cutoffs with few coefficients, at the cost of non-linear phase and possible instability.

In [ ]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
from audiodsp import add_noise, multi_sine

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

FS = 44100

## 1. Filter design and analysis
A filter is represented here either as FIR coefficients `b` or, for IIR filters, as **second-order sections** (`sos`): a cascade of 2nd-order filters (biquads). A high-order IIR filter written as a single pair of polynomials `(b, a)` is numerically fragile (section 4 shows why), so SciPy's IIR designs are used with `output='sos'`.

Four views describe a filter:
- **Magnitude response**: gain at each frequency, in dB.
- **Phase response**: phase shift at each frequency.
- **Group delay**: how long each frequency is delayed, $-d\varphi/d\omega$. Constant group delay (linear phase) keeps a waveform's shape. FIR filters with symmetric coefficients have it.
- **Impulse response**: the output for a single unit sample input.

In [ ]:
IIR_TYPES = {'Butterworth': 'butter', 'Chebyshev I': 'cheby1', 'Chebyshev II': 'cheby2', 'Elliptic': 'ellip'}

def design_filter(family, btype, f_low, f_high, fir_taps=101, iir_order=4, fs=FS):
    """Returns ('fir', b) or ('sos', sos)."""
    cutoff = f_low if btype in ('lowpass', 'highpass') else [f_low, f_high]
    if family == 'FIR (window)':
        if btype in ('highpass', 'bandstop') and fir_taps % 2 == 0:
            fir_taps += 1   # an even-length symmetric FIR always has a zero at fs/2
        return 'fir', signal.firwin(fir_taps, cutoff, pass_zero=btype, fs=fs)
    sos = signal.iirfilter(iir_order, cutoff, btype=btype, ftype=IIR_TYPES[family],
                           rp=1, rs=60, fs=fs, output='sos')
    return 'sos', sos

def frequency_response(filt, n_points=4096, fs=FS):
    kind, coeffs = filt
    if kind == 'fir':
        return signal.freqz(coeffs, worN=n_points, fs=fs)
    return signal.sosfreqz(coeffs, worN=n_points, fs=fs)

def group_delay(filt, n_points=4096, fs=FS):
    kind, coeffs = filt
    sections = [(coeffs, [1.0])] if kind == 'fir' else [(s[:3], s[3:]) for s in coeffs]
    total = 0
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')   # group delay is undefined exactly at a zero on the unit circle
        for b, a in sections:
            f, gd = signal.group_delay((b, a), w=n_points, fs=fs)
            total = total + gd
    return f, total

def apply_filter(filt, x):
    kind, coeffs = filt
    return signal.lfilter(coeffs, [1.0], x) if kind == 'fir' else signal.sosfilt(coeffs, x)

def plot_filter(filt, title, fs=FS, n_impulse=300):
    f, H = frequency_response(filt, fs=fs)
    mag_db = 20 * np.log10(np.abs(H) + 1e-12)
    _, gd = group_delay(filt, fs=fs)
    impulse = np.zeros(n_impulse)
    impulse[0] = 1
    h = apply_filter(filt, impulse)
    passband = mag_db > -3

    fig, axes = plt.subplots(2, 2, figsize=(15, 8))
    axes[0, 0].semilogx(f[1:], mag_db[1:])
    axes[0, 0].set_ylim(-100, 5)
    axes[0, 0].set_title('Magnitude response')
    axes[0, 0].set_ylabel('dB')
    axes[0, 1].semilogx(f[1:], np.degrees(np.unwrap(np.angle(H)))[1:])
    axes[0, 1].set_title('Phase response (unwrapped)')
    axes[0, 1].set_ylabel('degrees')
    axes[1, 0].semilogx(f[1:][passband[1:]], gd[1:][passband[1:]] / fs * 1000, '.', markersize=2)
    axes[1, 0].set_title('Group delay (passband only)')
    axes[1, 0].set_ylabel('ms')
    for ax in (axes[0, 0], axes[0, 1], axes[1, 0]):
        ax.set_xlim(20, fs / 2)
        ax.set_xlabel('Frequency (Hz)')
    axes[1, 1].stem(np.arange(n_impulse), h, basefmt=' ', markerfmt=' ')
    axes[1, 1].set_title('Impulse response')
    axes[1, 1].set_xlabel('Sample')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

rng = np.random.default_rng(0)
test_input = 0.5 * add_noise(multi_sine([110, 220, 440, 880, 1760, 3520, 7040], 2.0, FS) / 7, 0.15, rng)

def filter_tool(family='Butterworth', btype='lowpass', f_low=1000, f_high=4000, fir_taps=101, iir_order=4):
    if btype in ('bandpass', 'bandstop') and f_high <= f_low:
        print('For band-pass and band-stop, the upper cutoff must be above the lower cutoff.')
        return
    filt = design_filter(family, btype, f_low, f_high, fir_taps, iir_order)
    size = f'{len(filt[1])} taps' if filt[0] == 'fir' else f'order {iir_order} ({len(filt[1])} biquad sections)'
    cut = f'{f_low} Hz' if btype in ('lowpass', 'highpass') else f'{f_low}-{f_high} Hz'
    plot_filter(filt, f'{family} {btype}, {cut}, {size}')

    y = apply_filter(filt, test_input)
    fig, ax = plt.subplots(figsize=(15, 3.5))
    window = np.hanning(len(test_input))
    freqs = np.fft.rfftfreq(len(test_input), 1 / FS)
    for x, label in [(test_input, 'input: 7 octave-spaced tones + noise'), (y, 'filtered')]:
        ax.semilogx(freqs[1:], 20 * np.log10(np.abs(np.fft.rfft(x * window))[1:] + 1e-12), label=label)
    ax.set_xlim(20, FS / 2)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('dB')
    ax.set_title('Effect on the test signal')
    ax.legend()
    plt.show()
    print('Input:')
    display.display(display.Audio(test_input, rate=FS, normalize=False))
    print('Filtered:')
    display.display(display.Audio(np.clip(y, -1, 1), rate=FS, normalize=False))

display.display(interactive(
    filter_tool,
    family=widgets.Dropdown(options=['FIR (window)'] + list(IIR_TYPES), value='Butterworth', description='Family:'),
    btype=widgets.Dropdown(options=['lowpass', 'highpass', 'bandpass', 'bandstop'], description='Type:'),
    f_low=widgets.IntSlider(value=1000, min=50, max=15000, step=50, description='Cutoff (Hz):'),
    f_high=widgets.IntSlider(value=4000, min=100, max=20000, step=50, description='Upper (Hz):'),
    fir_taps=widgets.IntSlider(value=101, min=11, max=501, step=2, description='FIR taps:'),
    iir_order=widgets.IntSlider(value=4, min=1, max=12, description='IIR order:'),
))

Things to try:
- Compare **FIR** and **Butterworth** low-pass at 1 kHz. The FIR group delay is perfectly flat (linear phase) and the Butterworth's is not. Then compare their sizes: 101 FIR coefficients against 2 biquads of 5 coefficients each.
- Switch to **Elliptic**: the sharpest cutoff for a given order, with ripple in both bands and the most uneven group delay.
- Raise the IIR order and watch the impulse response ring longer. Sharp cutoffs always ring.

## 2. Real-time (block-based) filtering
An audio interface delivers samples in small blocks, for example 256 samples (5.8 ms at 44.1 kHz). The processing callback must finish each block before the next one arrives. A filter has memory of past inputs and outputs. To get the same result as filtering the whole signal at once, that memory, the **filter state**, must be carried from one block to the next.

The cell below filters a 220 Hz sawtooth (a bright, buzzy tone) block by block in two ways:
- **State carried over**: identical to offline filtering.
- **State reset every block**: each block starts from silence, which causes a glitch at every block boundary. You hear it as an extra buzz at $f_s / \text{block size}$ Hz.

It also times each block against the real-time budget.

In [ ]:
BLOCK_SIZE = 256
sos = signal.butter(4, 800, fs=FS, output='sos')
sawtooth = 0.5 * signal.sawtooth(2 * np.pi * 220 * np.arange(2 * FS) / FS)

def process_in_blocks(sos, x, block_size, carry_state=True):
    zi = np.zeros((sos.shape[0], 2))
    out = np.empty_like(x)
    block_times = []
    for start in range(0, len(x), block_size):
        block = x[start:start + block_size]
        t0 = time.perf_counter()
        y, zf = signal.sosfilt(sos, block, zi=zi)
        block_times.append(time.perf_counter() - t0)
        out[start:start + len(block)] = y
        zi = zf if carry_state else np.zeros_like(zi)
    return out, np.array(block_times)

offline = signal.sosfilt(sos, sawtooth)
carried, block_times = process_in_blocks(sos, sawtooth, BLOCK_SIZE, carry_state=True)
reset, _ = process_in_blocks(sos, sawtooth, BLOCK_SIZE, carry_state=False)

budget = BLOCK_SIZE / FS
print(f'Max difference, carried state vs offline: {np.max(np.abs(carried - offline)):.2e}')
print(f'Max difference, reset state vs offline:   {np.max(np.abs(reset - offline)):.2e}')
print(f'Block of {BLOCK_SIZE} samples = {budget * 1000:.2f} ms budget. '
      f'Median processing time {np.median(block_times) * 1e6:.0f} µs '
      f'({100 * np.median(block_times) / budget:.2f}% of the budget)')

view = slice(4 * BLOCK_SIZE - 150, 4 * BLOCK_SIZE + 150)
t_ms = np.arange(len(sawtooth))[view] / FS * 1000
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_ms, offline[view], linewidth=3, label='offline (whole signal)')
ax.plot(t_ms, carried[view], '--', label='blocks, state carried')
ax.plot(t_ms, reset[view], label='blocks, state reset')
ax.axvline(4 * BLOCK_SIZE / FS * 1000, color='k', linestyle=':', label='block boundary')
ax.set_xlabel('Time (ms)')
ax.set_title('Around a block boundary')
ax.legend(loc='upper right')
plt.show()

for name, y in [('Input sawtooth', sawtooth), ('Blocks, state carried', carried), ('Blocks, state reset', reset)]:
    print(f'{name}:')
    display.display(display.Audio(y, rate=FS))

## 3. Filter design comparison
Every design below meets the same low-pass specification at 44.1 kHz:
- passband 0 to 4 kHz, ripple at most 1 dB
- stopband from 5 kHz, attenuation at least 60 dB

For each FIR method the notebook searches for the smallest number of taps that meets the specification. For IIR filters, SciPy's order estimation functions (`buttord` and similar) give the minimum order directly.

| Method | Idea |
|---|---|
| Window (Kaiser) | Truncate the ideal (sinc) impulse response and taper it with a window |
| Frequency sampling | Specify the desired response at many frequencies, inverse-DFT it and window the result (`firwin2`) |
| Least squares | Minimize the total squared error against the desired response (`firls`) |
| Parks-McClellan | Minimize the *largest* error (equiripple, `remez`) |
| Butterworth | Maximally flat passband, no ripple anywhere |
| Chebyshev I / II | Ripple in the passband (I) or stopband (II) buys a sharper cutoff |
| Elliptic | Ripple in both bands. The lowest order possible for a given specification |

In [ ]:
F_PASS, F_STOP, RIPPLE_DB, ATTEN_DB = 4000, 5000, 1.0, 60.0
BANDS = [0, F_PASS, F_STOP, FS / 2]
# Passband ripple of 1 dB peak-to-peak allows a deviation of about 0.0575, the stopband 0.001.
# Weighting the stopband by their ratio tells the optimizers which error matters more.
STOP_WEIGHT = 0.0575 / 0.001

def measure(filt):
    f, H = frequency_response(filt, n_points=16384)
    mag = 20 * np.log10(np.abs(H) + 1e-12)
    ripple = mag[f <= F_PASS].max() - mag[f <= F_PASS].min()
    atten = -mag[f >= F_STOP].max()
    return ripple, atten

def meets_spec(filt):
    ripple, atten = measure(filt)
    return ripple <= RIPPLE_DB and atten >= ATTEN_DB

FIR_METHODS = {
    'Window (Kaiser)': lambda n: signal.firwin(n, (F_PASS + F_STOP) / 2, window=('kaiser', signal.kaiser_beta(ATTEN_DB)), fs=FS),
    # Unwindowed frequency sampling never reaches 60 dB here (about 33 dB at 161 taps). The window
    # fixes that but smooths the corners of the desired response outwards, so the ramp between
    # passband and stopband is specified inside the transition band, from 4.25 to 4.75 kHz.
    'Frequency sampling': lambda n: signal.firwin2(n, [0, 4250, 4750, FS / 2], [1, 1, 0, 0], window=('kaiser', signal.kaiser_beta(ATTEN_DB)), fs=FS),
    'Least squares': lambda n: signal.firls(n, BANDS, [1, 1, 0, 0], weight=[1, STOP_WEIGHT], fs=FS),
    'Parks-McClellan': lambda n: signal.remez(n, BANDS, [1, 0], weight=[1, STOP_WEIGHT], fs=FS),
}

designs, iir_orders = {}, {}
for name, make in FIR_METHODS.items():
    for n_taps in range(11, 601, 2):   # odd lengths: symmetric, linear-phase type I filters
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')   # remez can warn that it did not converge for small n
            filt = ('fir', make(n_taps))
        if meets_spec(filt):
            designs[name] = filt
            break

for name, (order_fn, design_fn) in {
    'Butterworth': (lambda: signal.buttord(F_PASS, F_STOP, RIPPLE_DB, ATTEN_DB, fs=FS),
                    lambda N, Wn: signal.butter(N, Wn, fs=FS, output='sos')),
    'Chebyshev I': (lambda: signal.cheb1ord(F_PASS, F_STOP, RIPPLE_DB, ATTEN_DB, fs=FS),
                    lambda N, Wn: signal.cheby1(N, RIPPLE_DB, Wn, fs=FS, output='sos')),
    'Chebyshev II': (lambda: signal.cheb2ord(F_PASS, F_STOP, RIPPLE_DB, ATTEN_DB, fs=FS),
                     lambda N, Wn: signal.cheby2(N, ATTEN_DB, Wn, fs=FS, output='sos')),
    'Elliptic': (lambda: signal.ellipord(F_PASS, F_STOP, RIPPLE_DB, ATTEN_DB, fs=FS),
                 lambda N, Wn: signal.ellip(N, RIPPLE_DB, ATTEN_DB, Wn, fs=FS, output='sos')),
}.items():
    N, Wn = order_fn()
    designs[name] = ('sos', design_fn(N, Wn))
    iir_orders[name] = N

print(f'{"Method":<20}{"Size":>14}{"Mult/sample":>13}{"Ripple dB":>11}{"Atten dB":>10}{"Delay spread ms":>17}')
for name, filt in designs.items():
    ripple, atten = measure(filt)
    f, gd = group_delay(filt)
    gd_pass = gd[f <= F_PASS] / FS * 1000
    if filt[0] == 'fir':
        size, mults = f'{len(filt[1])} taps', len(filt[1])
    else:
        size, mults = f'order {iir_orders[name]}', 5 * len(filt[1])
    print(f'{name:<20}{size:>14}{mults:>13}{ripple:>11.2f}{atten:>10.1f}{gd_pass.max() - gd_pass.min():>17.3f}')

"Mult/sample" counts multiplications per output sample: one per FIR tap, five per IIR biquad. A symmetric FIR can share multiplications between mirrored taps to halve its count. "Delay spread" is how much the group delay varies across the passband. It is 0 for linear-phase FIR filters, so every frequency is delayed equally, and grows for sharper IIR designs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for name, filt in designs.items():
    f, H = frequency_response(filt, n_points=8192)
    mag = 20 * np.log10(np.abs(H) + 1e-12)
    style = '-' if filt[0] == 'fir' else '--'
    axes[0].plot(f, mag, style, label=name)
    axes[1].plot(f, mag, style)
    fg, gd = group_delay(filt, n_points=8192)
    axes[2].plot(fg[fg <= F_PASS], gd[fg <= F_PASS] / FS * 1000, style)
axes[0].set_xlim(0, 10000)
axes[0].set_ylim(-100, 5)
axes[0].axvspan(F_PASS, F_STOP, color='0.9')
axes[0].axhline(-ATTEN_DB, color='k', linestyle=':')
axes[0].set_title('Magnitude (grey: transition band)')
axes[0].legend(fontsize=8)
axes[1].set_xlim(0, F_PASS * 1.05)
axes[1].set_ylim(-1.5, 0.5)
axes[1].set_title('Passband ripple (zoom)')
axes[2].set_title('Group delay in the passband')
axes[2].set_ylabel('ms')
for ax in axes:
    ax.set_xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

## 4. Poles, zeros and stability
A filter's transfer function $H(z) = B(z) / A(z)$ is fixed, apart from a gain, by its **zeros** (roots of $B$) and **poles** (roots of $A$). Evaluating $H(z)$ around the unit circle, $z = e^{j\omega}$, gives the frequency response. The response rises near a pole and dips near a zero.

A causal filter is **stable** only if every pole lies strictly inside the unit circle. FIR filters have no poles (apart from at the origin), so they are always stable.

The resonator below has a pair of poles at radius $r$ and angles $\pm\theta$, which sets the center frequency $f_c = \theta f_s / 2\pi$. It has zeros at $z = 1$ (DC) and $z = -1$ (Nyquist).
- $r$ close to 1 gives a tall, narrow peak and a long ringing impulse response.
- $r \ge 1$ makes the filter unstable: the impulse response grows instead of dying away.

In [ ]:
def plot_zplane(ax, zeros, poles):
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color='0.5', linestyle='--'))
    ax.plot(np.real(zeros), np.imag(zeros), 'o', markersize=9, fillstyle='none', label='zeros')
    ax.plot(np.real(poles), np.imag(poles), 'x', markersize=9, markeredgewidth=2, label='poles')
    lim = max(1.2, np.max(np.abs(np.concatenate([zeros, poles, [0]]))) * 1.1)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel('Real')
    ax.set_ylabel('Imaginary')
    ax.legend(loc='upper right')

def analyze_coefficients(b, a, title='', fs=FS, n_impulse=400):
    """Pole-zero plot, magnitude response and impulse response for any (b, a). Returns True if stable."""
    zeros, poles = np.roots(b), np.roots(a)
    # The tolerance stops a pole that is exactly on the unit circle (marginally stable,
    # rings forever) from being reported as stable because of rounding in np.roots
    stable = bool(np.all(np.abs(poles) < 1 - 1e-9))
    f, H = signal.freqz(b, a, worN=4096, fs=fs)
    impulse = np.zeros(n_impulse)
    impulse[0] = 1
    h = signal.lfilter(b, a, impulse)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    plot_zplane(axes[0], zeros, poles)
    axes[0].set_title(f'Pole-zero plot: {"STABLE" if stable else "UNSTABLE"}')
    axes[1].plot(f, 20 * np.log10(np.abs(H) + 1e-12))
    axes[1].set_title('Magnitude response')
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_ylabel('dB')
    axes[2].plot(h)
    axes[2].set_title('Impulse response')
    axes[2].set_xlabel('Sample')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f'Largest pole radius: {np.max(np.abs(poles)):.4f} -> {"stable" if stable else "unstable"}')
    return stable

def resonator(fc=2000, r=0.95):
    theta = 2 * np.pi * fc / FS
    b = [1.0, 0.0, -1.0]                                 # zeros at z = +1 and z = -1
    a = [1.0, -2 * r * np.cos(theta), r**2]              # poles at r * exp(+/- j theta)
    analyze_coefficients(b, a, f'Resonator: fc = {fc} Hz, r = {r}')

display.display(interactive(
    resonator,
    fc=widgets.IntSlider(value=2000, min=100, max=20000, step=100, description='fc (Hz):'),
    r=widgets.FloatSlider(value=0.95, min=0.5, max=1.02, step=0.005, readout_format='.3f', description='Pole radius r:'),
))

### Your own coefficients
`analyze_coefficients(b, a)` works on any filter. For an FIR filter, pass `a = [1]`. Edit the cell to try your own.

In [ ]:
analyze_coefficients([1, 2, 1], [1, -1.6, 0.81], 'Example: 2nd-order low-pass with poles at radius 0.9')

### Why second-order sections?
A high-order IIR filter can be written as one pair of long polynomials `(b, a)`, or as a cascade of biquads (`sos`). Mathematically they are the same filter. Numerically they are not. The poles of a narrow low-pass filter crowd together near $z = 1$, and the roots of a long polynomial are extremely sensitive to tiny rounding errors in its coefficients. Rounding can push a pole outside the unit circle.

Below, the same 10th-order Butterworth low-pass at 50 Hz is designed both ways. The `(b, a)` version is unstable. Each `sos` section has only two poles, so the rounding errors stay small.

In [ ]:
b_ba, a_ba = signal.butter(10, 50, fs=FS)
sos_10 = signal.butter(10, 50, fs=FS, output='sos')

stable_ba = analyze_coefficients(b_ba, a_ba, '10th-order Butterworth, 50 Hz, as a single (b, a)', n_impulse=20000)
sos_poles = np.concatenate([np.roots(s[3:]) for s in sos_10])
print(f'Same filter as {len(sos_10)} second-order sections: largest pole radius {np.max(np.abs(sos_poles)):.6f} '
      f'-> {"stable" if np.all(np.abs(sos_poles) < 1) else "unstable"}')